# Pyroptosis classification — one-shot Colab notebook

Trains our **ESM3 + attention-pooling** model to tell pyroptosis proteins from
non-pyroptosis, mirroring the ferroptosis pipeline: three tiers — **positive**
(pyroptosis), **housekeeping** (easy negatives, 10 functional categories), and
**hard-negative** (other cell-death: apoptosis + necroptosis + ferroptosis).
Redundancy is removed with **MMseqs2 at 0.3 identity**; generalization is measured
by **external validation on gene-disjoint held-out genes** (per-tier sensitivity/
specificity, exactly like the ferro figure).

**Before running:**
1. Upload the `Data/Pyroptosis/` folder to `MyDrive/JR_Ferro/Data/`.
2. Add a valid `HF_TOKEN` Colab secret (Runtime → Manage secrets; needs ESM3 access).
3. **Runtime → Run all.** Expect ~2–3 h (ESM3 embedding dominates). The last cell
   lists the files to download.


In [ ]:
# Install: etap (branch with ESM3 dtype fix) + deps + MMseqs2
!pip install -q --force-reinstall --no-deps \
  "git+https://github.com/Sitgttish/summer26.git@fix/esm3-bfloat16-dtype#subdirectory=eta_package"
!pip install -q esm biopython h5py scikit-learn openpyxl
# MMseqs2 static binary (avx2; if your Colab CPU lacks AVX2 use mmseqs-linux-sse41.tar.gz)
!wget -q https://mmseqs.com/latest/mmseqs-linux-avx2.tar.gz -O /content/mmseqs.tar.gz
!tar xzf /content/mmseqs.tar.gz -C /content
import subprocess
print(subprocess.run(['/content/mmseqs/bin/mmseqs','version'],capture_output=True,text=True).stdout.strip())
print('Install done.')

In [ ]:
# Drive, HuggingFace auth, paths, device
import os, re, gzip, io, time, random, subprocess
from pathlib import Path
import numpy as np, pandas as pd
from google.colab import drive; drive.mount('/content/drive')
from huggingface_hub import login, whoami
try:
    whoami(); print('HF: already authenticated')
except Exception:
    try:
        from google.colab import userdata; login(userdata.get('HF_TOKEN')); print('HF: via Colab secret')
    except Exception:
        login()
random.seed(42); np.random.seed(42)

PROJECT_ROOT = Path('/content/drive/MyDrive/JR_Ferro')
PYRO_DIR = PROJECT_ROOT/'Data/Pyroptosis'
OUT      = PROJECT_ROOT/'pyroptosis'; OUT.mkdir(parents=True, exist_ok=True)
LOCAL    = Path('/content/pyro'); LOCAL.mkdir(exist_ok=True)
CACHE    = LOCAL/'cache'    # ESM3 per-residue cache -> LOCAL disk (Drive FUSE unreliable; tens of GB)
MMSEQS   = '/content/mmseqs/bin/mmseqs'
VALID    = set('ACDEFGHIKLMNPQRSTVWY')
assert PYRO_DIR.exists(), f'Upload Data/Pyroptosis to {PYRO_DIR} first.'
print('data dir:', PYRO_DIR)

In [ ]:
# Parse xlsx/fasta -> per-tier (gene, sequence) -> raw FASTAs
def read_xlsx_any(path):
    raw = Path(path).read_bytes()
    if raw[:2]==b'\x1f\x8b': raw = gzip.decompress(raw)
    return pd.read_excel(io.BytesIO(raw), engine='openpyxl')
def gene_of(fname):
    s=re.sub(r'(\.xlsx)+$','',fname); s=re.sub(r'^uniparc_','',s)
    return re.split(r'_AND_|_20\d\d', s)[0].strip('_')
def clean(s): return ''.join(c for c in str(s).upper() if c in VALID)
def seqs_from_xlsx(path):
    df=read_xlsx_any(path); df.columns=[c.strip() for c in df.columns]
    col=next((c for c in df.columns if 'seq' in c.lower()), df.columns[-1])
    return [clean(v) for v in df[col] if len(clean(v))>=10]

records=[]   # (tier, gene, seq)
for f in sorted((PYRO_DIR/'positive').glob('*.xlsx')):
    g=gene_of(f.name)
    for s in seqs_from_xlsx(f): records.append(('positive',g,s))
for f in sorted((PYRO_DIR/'easy_negative').glob('*/*.xlsx')):
    g=gene_of(f.name)
    for s in seqs_from_xlsx(f): records.append(('housekeeping',g,s))
for f in sorted((PYRO_DIR/'hard_negative').glob('*/*.xlsx*')):
    g=gene_of(f.name)
    for s in seqs_from_xlsx(f): records.append(('hardneg',g,s))
ferro_fa=PYRO_DIR/'hard_negative'/'ferroptosis_rep.fasta'
if ferro_fa.exists():
    g=None; buf=[]
    for line in open(ferro_fa):
        if line.startswith('>'):
            if g and buf:
                s=clean(''.join(buf))
                if len(s)>=10: records.append(('hardneg',g,s))
            m=re.search(r'gene=([^|]*)',line); g=m.group(1).split('_AND_')[0] if m else 'FERRO'; buf=[]
        else: buf.append(line.strip())
    if g and buf:
        s=clean(''.join(buf))
        if len(s)>=10: records.append(('hardneg',g,s))

raw=pd.DataFrame(records, columns=['tier','gene','seq'])
print('RAW (before MMseqs2):')
print(raw.groupby('tier').agg(seqs=('seq','size'), genes=('gene','nunique')).to_string())

LOCAL_RAW={}
for tier in ['positive','housekeeping','hardneg']:
    p=LOCAL/f'{tier}_raw.fasta'; LOCAL_RAW[tier]=p
    sub=raw[raw.tier==tier].reset_index(drop=True)
    with open(p,'w') as fo:
        for i,r in sub.iterrows(): fo.write(f'>{r.gene}__{tier}__{i}\n{r.seq}\n')
print('raw FASTAs written')

In [ ]:
# MMseqs2 easy-cluster at 0.3 identity, per tier
def mmseqs_rep(rawfa, prefix):
    subprocess.run([MMSEQS,'easy-cluster',str(rawfa),str(prefix),str(prefix)+'_tmp',
        '--min-seq-id','0.3','-c','0.8','--cov-mode','1'],
        check=True, stdout=subprocess.DEVNULL)
    return Path(str(prefix)+'_rep_seq.fasta')
reps=[]
for tier in ['positive','housekeeping','hardneg']:
    rp=mmseqs_rep(LOCAL_RAW[tier], LOCAL/f'{tier}_clust')
    g=None; buf=[]
    for line in open(rp):
        if line.startswith('>'):
            if g and buf: reps.append((tier,g,''.join(buf)))
            g=line[1:].strip().split()[0].split('__')[0]; buf=[]
        else: buf.append(line.strip())
    if g and buf: reps.append((tier,g,''.join(buf)))
rep=pd.DataFrame(reps, columns=['tier','gene','seq'])
print('AFTER MMseqs2 0.3:')
print(rep.groupby('tier').agg(seqs=('seq','size'), genes=('gene','nunique')).to_string())

In [ ]:
# Cap per gene + gene-disjoint 15% external split; write etap FASTAs
MAX_PER_GENE = 1200        # bounds the ESM3 cache size; lower if Colab disk is tight
EV_FRAC = 0.15
rep = rep.groupby('gene', group_keys=False).apply(
        lambda g: g.sample(min(len(g), MAX_PER_GENE), random_state=42)).reset_index(drop=True)

gene_tier={}; split={}
for tier in ['positive','housekeeping','hardneg']:
    genes=sorted(rep[rep.tier==tier].gene.unique()); random.Random(42).shuffle(genes)
    n_ev=max(2, round(len(genes)*EV_FRAC)); ev=set(genes[:n_ev])
    for gname in genes:
        gene_tier[gname]=tier; split[gname]='external' if gname in ev else 'train'
rep['split']=rep.gene.map(split)
rep['label']=rep.tier.map({'positive':1,'housekeeping':0,'hardneg':0})
pd.DataFrame({'gene':list(gene_tier),'tier':[gene_tier[g] for g in gene_tier],
             'split':[split[g] for g in gene_tier]}).to_csv(OUT/'pyro_gene_tier.csv', index=False)

def write_fa(path, sub):
    with open(path,'w') as fo:
        for i,r in sub.reset_index(drop=True).iterrows():
            fo.write(f'>{r.gene}__{r.tier}__{i}|label={r.label}\n{r.seq}\n')
TR=rep[rep.split=='train']; EV=rep[rep.split=='external']
POS_FA=str(LOCAL/'train_pos.fasta'); NEG_FA=str(LOCAL/'train_neg.fasta'); EV_FA=str(LOCAL/'ev_labeled.fasta')
write_fa(POS_FA, TR[TR.label==1]); write_fa(NEG_FA, TR[TR.label==0]); write_fa(EV_FA, EV)
print('TRAIN  pos', int((TR.label==1).sum()), ' neg', int((TR.label==0).sum()),
      ' | genes', TR.gene.nunique())
print('EXTERNAL', len(EV), 'seqs:')
print(EV.groupby('tier').agg(seqs=('seq','size'), genes=('gene','nunique')).to_string())

In [ ]:
# Train (etap): random 64/16/20 split of TRAIN; ESM3 cache on local disk
OUTD=OUT/'model'
def _done(p):
    import torch
    try: return p.exists() and 'test_metrics' in torch.load(p,map_location='cpu',weights_only=False)
    except Exception: return False
if _done(OUTD/'best_model.pth'):
    print('Trained model already on Drive — skipping training. Delete',
          OUTD/'best_model.pth','to retrain.')
else:
    get_ipython().system(f'etap --train "{POS_FA}" "{NEG_FA}" "{OUTD}/" '
                         f'--cache-dir "{CACHE}" --embed-batch-size 8')
print('model dir:', os.listdir(OUTD) if OUTD.exists() else '(none)')

In [ ]:
# Internal TEST set: reconstruct the EXACT held-out split (no retraining)
# `etap --train` used a stratified 20% test split (seed 42) and stored the exact
# test indices in the checkpoint metadata. We rebuild that precise test set and
# evaluate the ALREADY-trained model on it — pure inference, no retraining.
from etap.data import parse_fasta
from sklearn.model_selection import train_test_split
import torch

OUTD = OUT / 'model'
CKPT = str(OUTD / 'best_model.pth')
assert Path(CKPT).exists(), f'No trained model at {CKPT} — run the training cell first.'
TEST_FASTA = str(LOCAL / 'internal_test.fasta')

pos_rec = parse_fasta(POS_FA); neg_rec = parse_fasta(NEG_FA)
all_rec = pos_rec + neg_rec
y   = np.array([1] * len(pos_rec) + [0] * len(neg_rec))
idx = np.arange(len(y))

meta = torch.load(CKPT, map_location='cpu', weights_only=False).get('metadata', {})
aligned = ('genes' in meta and 'labels' in meta and len(meta['genes']) == len(all_rec)
           and all(all_rec[i][1] == meta['genes'][i] for i in range(len(all_rec)))
           and all(int(y[i]) == int(meta['labels'][i]) for i in range(len(all_rec))))
if aligned and 'idx_test' in meta:
    idx_test = np.array(meta['idx_test'])
    print(f'Test set matches the checkpoint exactly — {len(idx_test)} seqs.')
else:
    _, idx_test = train_test_split(idx, test_size=0.20, stratify=y, random_state=42)
    why = 'no idx_test in checkpoint' if 'idx_test' not in meta \
          else 'metadata did not align (check POS_FA/NEG_FA)'
    print(f'WARNING: using reproduced 20% split ({len(idx_test)} seqs) — {why}.')

with open(TEST_FASTA, 'w') as f:
    for i in idx_test:
        _, gene, seq, _ = all_rec[i]
        f.write(f'>{gene}|label={int(y[i])}\n{seq}\n')
print(f'Internal TEST: {len(idx_test)} seqs '
      f'({int(y[idx_test].sum())} pos / {int((y[idx_test] == 0).sum())} neg), '
      f'{len({all_rec[i][1] for i in idx_test})} genes  ->  {TEST_FASTA}')


In [ ]:
# Inference on the internal test set -> per-gene performance (no retraining)
# etap --eval only loads ESM3 to embed these test sequences, then runs the trained
# model. Output CSV columns: header, gene, prob_positive, predicted_label, true_label, correct.
TEST_PRED = str(OUT / 'pyro_test_predictions.csv')
get_ipython().system(f'etap --eval "{CKPT}" "{TEST_FASTA}" "{TEST_PRED}" '
                     f'--gene-analyze --analyze-dir "{OUT}/test_analysis/"')

df = pd.read_csv(TEST_PRED)
tier_map = pd.read_csv(OUT / 'pyro_gene_tier.csv').set_index('gene')['tier']
pg = (df.groupby('gene')
        .apply(lambda x: pd.Series({
            'n': len(x), 'label': int(x['true_label'].iloc[0]),
            'acc': (x['predicted_label'] == x['true_label']).mean(),
            'mean_prob': x['prob_positive'].mean()}))
        .reset_index())
pg['tier']   = pg['gene'].map(tier_map)
pg['metric'] = np.where(pg['label'] == 1, 'sensitivity', 'specificity')
pg = pg[['gene', 'tier', 'label', 'metric', 'n', 'acc', 'mean_prob']]
pg.to_csv(OUT / 'pyro_test_pergene.csv', index=False)

posg, negg = pg[pg.label == 1], pg[pg.label == 0]
print(f'\nInternal-test per-gene  ->  {OUT / "pyro_test_pergene.csv"}  ({len(pg)} genes)')
print(f'  positive macro-acc {posg["acc"].mean():.3f} (n={len(posg)})  |  '
      f'negative macro-acc {negg["acc"].mean():.3f} (n={len(negg)})')
for t, grp in pg.groupby('tier'):
    print(f'    {t:12s}: macro-acc {grp["acc"].mean():.3f}  (n={len(grp)} genes)')


In [ ]:
# External validation (etap --eval on held-out genes)
CKPT=str(OUTD/'best_model.pth'); PREDS=str(OUT/'pyro_ev_predictions.csv')
get_ipython().system(f'etap --eval "{CKPT}" "{EV_FA}" "{PREDS}" '
                     f'--gene-analyze --analyze-dir "{OUT}/ev_analysis/"')

In [ ]:
# Per-tier metrics (mirrors the ferro external-validation analysis)
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score
gt=pd.read_csv(OUT/'pyro_gene_tier.csv').set_index('gene')['tier'].to_dict()
pred=pd.read_csv(OUT/'pyro_ev_predictions.csv')
pred['tier']=pred.gene.map(gt)
pred.to_csv(OUT/'pyro_ev_predictions.csv', index=False)
yt=pred.true_label.values; yp=pred.prob_positive.values
def frac_pos(m): return float((pred[m].prob_positive>=.5).mean())
summary={'n':len(pred),
  'AUROC': round(roc_auc_score(yt,yp),4),
  'AP': round(average_precision_score(yt,yp),4),
  'accuracy': round(accuracy_score(yt,pred.predicted_label),4),
  'sensitivity_positive': round(frac_pos(pred.tier=='positive'),4),
  'specificity_housekeeping': round(1-frac_pos(pred.tier=='housekeeping'),4),
  'specificity_hardneg': round(1-frac_pos(pred.tier=='hardneg'),4)}
pd.DataFrame([summary]).to_csv(OUT/'pyro_ev_tier_metrics.csv', index=False)
print('EXTERNAL VALIDATION — per tier')
for k,v in summary.items(): print(f'  {k:26s}: {v}')
pg=pred.groupby(['tier','gene']).agg(n=('prob_positive','size'),
     frac_pred_pos=('prob_positive',lambda s:float((s>=.5).mean())),
     mean_prob=('prob_positive','mean')).reset_index()
pg.to_csv(OUT/'pyro_ev_pergene.csv', index=False)
print('\nper-gene table ->', OUT/'pyro_ev_pergene.csv', '(', len(pg),'genes )')

In [ ]:
# Files to download
print('=== INTERNAL TEST (etap training, random split) ===')
tm=OUTD/'test_metrics.csv'
if tm.exists(): print(pd.read_csv(tm).to_string(index=False))
print('\n===============  DOWNLOAD THESE  ===============')
for f in ['model/test_metrics.csv','pyro_ev_predictions.csv','pyro_ev_tier_metrics.csv',
          'pyro_ev_pergene.csv','pyro_gene_tier.csv',
          'pyro_test_predictions.csv','pyro_test_pergene.csv']:
    p=OUT/f; print('  ', str(p), ('(%.1f KB)'%(p.stat().st_size/1024)) if p.exists() else '(MISSING!)')
print('===============================================')
print('Folder:', OUT)

**Send me:** `pyro_test_pergene.csv`, `pyro_test_predictions.csv`, `pyro_ev_predictions.csv`, `pyro_ev_tier_metrics.csv`,
`pyro_ev_pergene.csv`, `model/test_metrics.csv`, and `pyro_gene_tier.csv`. I'll build
the pyroptosis figure (internal vs external, per-tier specificity) in the paper style,
as a second generalization example alongside senescence.